In [ ]:
import os
import sys
import difflib
import collections
import re 

# Import from pds_parser 
from pds_parser import (
    PdsParser, PdsBlock, PdsKeyValuePair, PdsList, 
    PdsComment, PdsBlankLine, PdsOperatorCondition, PdsNode
)
# Import from pds_differ
from pds_differ import PdsDiffer, PdsChange, _find_node_by_diff_path_in_tree, g_subsumed_mod_block_paths_for_simulation # Import find helper and global state

print(f"--- Successfully imported PdsParser from: {PdsParser.__module__}.py ---")
print(f"--- Successfully imported PdsDiffer from: {PdsDiffer.__module__}.py ---")
print("-" * 80)

# --- Paths ---
# Keep existing real file paths for reference, but comment out test calls
# SIEGE_EVENTS_MOD_PATH = r"C:\Users\Galaxy\Documents\Paradox Interactive\Crusader Kings III\mod\custom_changes\events\siege_events.txt"
# SIEGE_EVENTS_OLD_VANILLA_PATH = r"C:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\old_ver\events\siege_events.txt"
# SIEGE_EVENTS_NEW_VANILLA_PATH = r"C:\Program Files (x86)\Steam\steamapps\common\Crusader Kings III\game\events\siege_events.txt"

# INNOVATIONS_MOD_PATH = r"C:\Users\Galaxy\Documents\Paradox Interactive\Crusader Kings III\mod\custom_changes\common\culture\innovations\00_tribal_innovations.txt"
# INNOVATIONS_OLD_VANILLA_PATH = r"C:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\old_ver\common\culture\innovations\00_tribal_innovations.txt"
# INNOVATIONS_NEW_VANILLA_PATH = r"C:\Program Files (x86)\Steam\steamapps\common\Crusader Kings III\game\common\culture\innovations\00_tribal_innovations.txt"

# TRAITS_MOD_PATH = r"C:\Users\Galaxy\Documents\Paradox Interactive\Crusader Kings III\mod\custom_changes\common\traits\00_traits.txt"
# TRAITS_OLD_VANILLA_PATH = r"C:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\old_ver\common\traits\00_traits.txt"
# TRAITS_NEW_VANILLA_PATH = r"C:\Program Files (x86)\Steam\steamapps\common\Crusader Kings III\game\common\traits\00_traits.txt"

# --- Benchmark File Paths ---
BENCHMARK_DIR = os.path.join(os.getcwd(), "benchmark")
BENCHMARK_OLD = os.path.join(BENCHMARK_DIR, "benchmark_test_old.txt")
BENCHMARK_MOD = os.path.join(BENCHMARK_DIR, "benchmark_test_mod.txt")
BENCHMARK_NEW = os.path.join(BENCHMARK_DIR, "benchmark_test_new.txt")

# Ensure benchmark directory exists
os.makedirs(BENCHMARK_DIR, exist_ok=True)

# --- File I/O Helpers (UNCHANGED) ---
def get_file_content(filepath):
    try:
        with open(filepath, 'r', encoding='utf-8-sig') as f: return f.read()
    except UnicodeDecodeError:
        try:
            with open(filepath, 'r', encoding='utf-8') as f: return f.read()
        except Exception as e_inner: sys.stderr.write(f"ERROR reading {filepath} (fallback): {e_inner}\n"); return None
    except FileNotFoundError: sys.stderr.write(f"WARNING: File not found: {filepath}\n"); return None
    except Exception as e: sys.stderr.write(f"ERROR reading {filepath}: {e}\n"); return None

def write_to_file(filepath, content):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    try:
        with open(filepath, 'w', encoding='utf-8-sig') as f: f.write(content); return True
    except Exception as e: sys.stderr.write(f"ERROR writing to {filepath}: {e}\n"); return False

def normalize_for_comparison(text):
    if not text: return ""
    return re.sub(r'\n(\s*\n)+', '\n', text).strip()

# --- Assertion Helper ---
def assert_node_value(root_nodes_list, diff_path, expected_value=None, expected_type=None, check_comment_substring=None):
    """
    Finds a node by its diff_path in the list of root_nodes and asserts its properties.
    diff_path should be a list of segments like ['block_key___0', 'child_key___0'].
    Expected value can be the primitive value or a tuple for complex nodes.
    check_comment_substring checks if the node's comment_text_on_line *contains* the substring
    or if, for a block, any child PdsComment's text contains the substring.
    Returns True if all assertions pass for this node, False otherwise.
    """
    path_str = '->'.join(diff_path)
    node = _find_node_by_diff_path_in_tree(root_nodes_list, diff_path)
    
    success = True
    messages = []

    if node is None:
        messages.append(f"FAIL: Node not found at path '{path_str}'.")
        success = False
    else:
        if expected_type is not None and not isinstance(node, expected_type):
             messages.append(f"FAIL: Node at '{path_str}' has incorrect type. Expected {expected_type.__name__}, got {type(node).__name__}.")
             success = False

        actual_value = None
        # Determine the actual value based on node type for comparison
        if isinstance(node, (PdsKeyValuePair, PdsOperatorCondition)):
            actual_value = node.value
        elif isinstance(node, PdsList):
            # For lists, we might check children structure later or in separate assertions.
            # Just indicate it's a list node itself.
            actual_value = "<PdsList>" # Sentinel value for existence/type check
        elif isinstance(node, PdsBlock):
             # For blocks, check key if expected_value is provided, otherwise sentinel for existence/type.
             if expected_value is not None and not isinstance(expected_value, str): # Expected value is a primitive for Block key check
                 messages.append(f"ASSERTION ERROR: Invalid expected_value type for PdsBlock key check: {type(expected_value)}. Must be str or None.")
                 success = False # This is an error in the assertion itself
             else:
                actual_value = node.key # Compare block key
                if expected_value is not None and actual_value != expected_value:
                     messages.append(f"FAIL: Block key mismatch at '{path_str}'. Expected '{expected_value}', got '{actual_value}'.")
                     success = False
                elif expected_value is None:
                    actual_value = "<PdsBlock>" # Sentinel value for existence/type check
                    
        elif isinstance(node, PdsComment):
            actual_value = node.comment_text
        elif isinstance(node, PdsBlankLine):
             actual_value = "BLANK" # Sentinel value

        # Compare value/structure based on node type and expected_value type
        if expected_value is not None:
             if isinstance(node, (PdsKeyValuePair, PdsOperatorCondition, PdsComment)):
                 if actual_value != expected_value:
                     messages.append(f"FAIL: Value mismatch at '{path_str}'. Expected '{expected_value}', got '{actual_value}'.")
                     success = False
             elif isinstance(node, PdsList) and isinstance(expected_value, (list, tuple)):
                 # Compare list contents deeply. Convert nodes within lists to their structural components for comparison.
                 actual_list_content = [v.get_structural_components(shallow_block=False) if isinstance(v, PdsNode) else v for v in node.values]
                 expected_list_content = [v.get_structural_components(shallow_block=False) if isinstance(v, PdsNode) else v for v in expected_value]
                 if actual_list_content != expected_list_content:
                      messages.append(f"FAIL: List content mismatch at '{path_str}'.\n  Expected: {expected_list_content}\n  Got:      {actual_list_content}")
                      success = False
             # Block key comparison handled above

        # Check for comment substring
        if check_comment_substring is not None:
            comment_found = False
            if hasattr(node, 'comment_text_on_line') and node.comment_text_on_line and check_comment_substring in node.comment_text_on_line:
                comment_found = True
            # For blocks, also check child PdsComment nodes
            if isinstance(node, PdsBlock):
                 if any(isinstance(child, PdsComment) and child.comment_text and check_comment_substring in child.comment_text for child in node.children):
                      comment_found = True

            if not comment_found:
                 node_comment_line = getattr(node, 'comment_text_on_line', 'N/A')
                 child_comments_text = [c.comment_text for c in node.children if isinstance(c, PdsComment)] if isinstance(node, PdsBlock) else []
                 messages.append(f"FAIL: Comment substring '{check_comment_substring}' not found for '{path_str}'. Line comment: '{node_comment_line}'. Child comments: {child_comments_text}.")
                 success = False
        
        # --- Additional Checks for Absence ---
        # If expected_value is None, it usually means the node should *not* exist.
        # This check is done implicitly by `node is None` at the start.
        # Let's make an explicit case for asserting absence.
    
    # --- Report Result ---
    if success and node is not None:
         print(f"  PASS: Node at '{path_str}' matches expectations.")
    elif success and node is None and expected_value is None: # Correctly absent
         print(f"  PASS: Node at '{path_str}' is correctly absent.")
    else: # Failure (or unexpected absence/presence)
        # messages already populated or node not found/unexpectedly found
        if messages: # Messages from checks if node was found
             for msg in messages:
                 print(f"  {msg}")
        elif node is not None and expected_value is None:
             print(f"  FAIL: Node at '{path_str}' was found, but expected to be absent.")
        # else: messages should cover the node is None case from the top

    return success

def assert_node_absent(root_nodes_list, diff_path):
    """Asserts that a node does NOT exist at the given path."""
    path_str = '->'.join(diff_path)
    node = _find_node_by_diff_path_in_tree(root_nodes_list, diff_path)
    if node is None:
        print(f"  PASS: Node at '{path_str}' is correctly absent.")
        return True
    else:
        print(f"  FAIL: Node at '{path_str}' was found, but expected to be absent. Found node: {node}")
        return False


def run_benchmark_test(test_name, old_path, mod_path, new_path, assertions_func):
    # Reset global state managed by PdsDiffer for simulation
    global g_subsumed_mod_block_paths_for_simulation # Need to access the global variable
    g_subsumed_mod_block_paths_for_simulation = set() # Ensure it's clean for the assertions check later

    print(f"\n{'='*20} RUNNING BENCHMARK TEST: {test_name} {'='*20}")
    safe_test_name = "".join(c if c.isalnum() or c in (' ', '_', '-') else '_' for c in test_name)
    safe_test_name = safe_test_name.replace(" ", "_")
    
    test_output_dir = os.path.join(os.getcwd(), "test_output", "benchmark", safe_test_name)
    os.makedirs(test_output_dir, exist_ok=True)
    print(f"Output files for this test will be saved to: {test_output_dir}")

    old_content_raw = get_file_content(old_path)
    mod_content_raw = get_file_content(mod_path)
    new_content_raw = get_file_content(new_path)

    if old_content_raw is None and mod_content_raw is None and new_content_raw is None:
        print(f"SKIPPING TEST '{test_name}': All three input files are missing or could not be read.")
        return
    
    write_to_file(os.path.join(test_output_dir, os.path.basename(old_path).replace(".txt", "_OLD_RAW.txt")), old_content_raw or "")
    write_to_file(os.path.join(test_output_dir, os.path.basename(mod_path).replace(".txt", "_MOD_RAW.txt")), mod_content_raw or "")
    write_to_file(os.path.join(test_output_dir, os.path.basename(new_path).replace(".txt", "_NEW_RAW.txt")), new_content_raw or "")
    print(f"  Raw files saved.")

    parser = PdsParser() 
    print(f"  Parsing Old file: {os.path.basename(old_path)}")
    old_nodes = parser.parse_file(old_path) if old_content_raw is not None else []
    print(f"  Parsing Mod file: {os.path.basename(mod_path)}")
    mod_nodes = parser.parse_file(mod_path) if mod_content_raw is not None else []
    print(f"  Parsing New file: {os.path.basename(new_path)}")
    new_nodes = parser.parse_file(new_path) if new_content_raw is not None else []
    
    if not (old_nodes or mod_nodes or new_nodes):
         print(f"SKIPPING TEST '{test_name}': No nodes parsed from any input file.")
         return

    reconstructed_old = PdsParser._nodes_to_string(old_nodes)
    reconstructed_mod = PdsParser._nodes_to_string(mod_nodes)
    reconstructed_new = PdsParser._nodes_to_string(new_nodes)
    write_to_file(os.path.join(test_output_dir, os.path.basename(old_path).replace(".txt", "_OLD_RECONSTRUCTED.txt")), reconstructed_old)
    write_to_file(os.path.join(test_output_dir, os.path.basename(mod_path).replace(".txt", "_MOD_RECONSTRUCTED.txt")), reconstructed_mod)
    write_to_file(os.path.join(test_output_dir, os.path.basename(new_path).replace(".txt", "_NEW_RECONSTRUCTED.txt")), reconstructed_new)
    print(f"  Reconstructed files saved.")
    
    normalized_old_raw = normalize_for_comparison(old_content_raw or "")
    normalized_mod_raw = normalize_for_comparison(mod_content_raw or "")
    normalized_new_raw = normalize_for_comparison(new_content_raw or "")

    if old_content_raw and normalize_for_comparison(reconstructed_old) != normalized_old_raw: print(f"\nWARNING: Normalized reconstruction mismatch for OLD file: {os.path.basename(old_path)}.")
    if mod_content_raw and normalize_for_comparison(reconstructed_mod) != normalized_mod_raw: print(f"\nWARNING: Normalized reconstruction mismatch for MOD file: {os.path.basename(mod_path)}.")
    if new_content_raw and normalize_for_comparison(reconstructed_new) != normalized_new_raw: print(f"\nWARNING: Normalized reconstruction mismatch for NEW file: {os.path.basename(new_path)}.")

    differ = PdsDiffer()
    changes = differ.diff_nodes(old_nodes, mod_nodes, new_nodes)
    print(f"\n--- DETECTED CHANGES for '{test_name}' ({len(changes)} changes) ---")
    if not changes: print("    No significant changes detected by PdsDiffer.")
    max_changes_to_print = 50 # Limit printing for large diffs
    for i, change in enumerate(changes):
        if i < max_changes_to_print: print(change)
        elif i == max_changes_to_print: print(f"    ... (omitting {len(changes) - max_changes_to_print} more changes)"); break

    print(f"\n--- Test Script: Requesting PdsDiffer to SIMULATE MERGE for '{test_name}' ---")
    # Simulate merge using the differ's internal logic
    simulated_merged_nodes_root_list = differ.simulate_three_way_merge(changes, new_nodes)

    simulated_merged_content = PdsParser._nodes_to_string(simulated_merged_nodes_root_list)
    merged_filename_base = os.path.basename(new_path).replace(".txt", "") if new_path else "unknown_new_file"
    sim_merged_filepath = os.path.join(test_output_dir, f"{merged_filename_base}_SIMULATED_MERGED.txt")
    write_to_file(sim_merged_filepath, simulated_merged_content)
    print(f"\n  Simulated merged content saved by Test Script to: {os.path.basename(sim_merged_filepath)}")
    
    print(f"\n--- Asserting Specific Merge Outcomes for '{test_name}' ---")
    # Pass the simulated merged nodes list to the assertions function
    assertions_func(simulated_merged_nodes_root_list)
    print("--- Assertions Complete ---")

    print(f"\n--- DIFF: SIMULATED MERGED vs NORMALIZED NEW VANILLA RAW for '{test_name}' ---")
    diff_filename = os.path.join(test_output_dir, f"{merged_filename_base}_MERGED_VS_NEW_RAW.diff")
    normalized_simulated_merged = normalize_for_comparison(simulated_merged_content)
    
    diff_lines_exist = False
    if new_content_raw is not None: 
        with open(diff_filename, 'w', encoding='utf-8') as f_diff:
            diff_lines = list(difflib.unified_diff(
                normalized_new_raw.splitlines(keepends=True),
                normalized_simulated_merged.splitlines(keepends=True),
                fromfile='NORMALIZED_NEW_VANILLA_RAW', tofile='NORMALIZED_SIMULATED_MERGED', lineterm=''))
            if diff_lines: 
                f_diff.writelines(diff_lines)
                print(f"  Diff (Normalized) saved to: {os.path.basename(diff_filename)}")
                diff_lines_exist = True
            else: 
                print("  NORMALIZED SIMULATED MERGED is identical to NORMALIZED NEW VANILLA RAW.")
    else:
        print(f"  Skipping diff generation because NEW vanilla raw content was not available for '{test_name}'.")

    if not diff_lines_exist and new_content_raw is not None:
         print("  CONFIRMED: NORMALIZED SIMULATED MERGED is identical to NORMALIZED NEW VANILLA RAW.")
    elif new_content_raw is not None:
         print(f"  ATTENTION: Diff found between NORMALIZED SIMULATED MERGED and NEW VANILLA RAW. Review '{os.path.basename(diff_filename)}'.")

    print("-" * 80)


# --- Define Benchmark Assertions Function ---
def benchmark_assertions(merged_nodes):
    print("\n--- Running Specific Benchmark Assertions ---")
    # We need line numbers from the OLD file to construct diff paths based on initial indices.
    # The line numbers in the benchmark files are added as comments like # LXXX
    # These need to be manually derived or stored programmatically if files change.
    # For this static benchmark, I'll use hardcoded line numbers inferred from the text.
    # The _find_node_by_diff_path_in_tree uses the key and OCCURRENCE index (___0, ___1 etc),
    # not line number directly, unless the node type fallback uses line number.
    # So paths like 'primitive_string___0' should work if it's the first node with that key.
    # For anonymous blocks in lists, the path segment uses class name and line number: '__BLOCK_LXXX___0'.
    # Let's re-check the line numbers in the OLD benchmark file:
    # list_with_anon_blocks first block: L88 -> __BLOCK_L88___0
    # list_with_anon_blocks second block: L89 -> __BLOCK_L89___0
    # complex_list_items anon block: L188 -> __BLOCK_L188___0
    # random_list_benchmark 10 block: L198 -> 10___0
    # random_list_benchmark 20 block: L203 -> 20___0
    # random_list_benchmark 30 block: L206 -> 30___0
    # random_list_benchmark 40 block: L209 -> 40___0
    # random_list_benchmark 50 block: L212 -> 50___0


    # Section 1: Primitive Value Changes & Conflicts
    assert_node_value(merged_nodes, ['primitive_string___0'], expected_value="mod_string_changed", expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:CONFLICT_MODIFIED")
    assert_node_value(merged_nodes, ['primitive_number___0'], expected_value=999.9, expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:MOD_MODIFIED")
    assert_node_value(merged_nodes, ['primitive_bool___0'], expected_value=False, expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:MOD_MODIFIED")
    assert_node_value(merged_nodes, ['primitive_float___0'], expected_value=2.0, expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:VANILLA_MODIFIED") # Mod kept, Vanilla modified

    # Section 2: Additions, Deletions, Unchanged
    # delete_mod_modify_new: CONFLICT_DELETION_MOD_DELETED_VANILLA_MODIFIED -> Vanilla Modified (kept)
    # The change type implies Mod deleted the OLD version, Vanilla modified the OLD version.
    # The merge resolution is currently CONFLICT_DELETION_MOD_DELETED_VANILLA_MODIFIED which means Vanilla's modified version is KEPT.
    assert_node_value(merged_nodes, ['delete_mod_modify_new___0'], expected_type=PdsBlock, check_comment_substring="SimMerge:CONFLICT_DELETION_MOD_DELETED_VANILLA_MODIFIED_VANILLA_MOD_KEPT")
    assert_node_value(merged_nodes, ['delete_mod_modify_new___0', 'value___0'], expected_value=2, expected_type=PdsKeyValuePair, check_comment_substring=None) # Vanilla modified value
    assert_node_value(merged_nodes, ['delete_mod_modify_new___0', 'keep_me___0'], expected_value=True, expected_type=PdsKeyValuePair, check_comment_substring=None) # Kept identical child
    assert_node_value(merged_nodes, ['delete_mod_modify_new___0', 'vanilla_added_child___0'], expected_value=False, expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:VANILLA_ADDED") # Vanilla added child

    # add_mod_only: MOD_ADDED -> Should be present (find by key)
    root_container = merged_nodes
    mod_added_kvp = None
    for node in root_container:
        if isinstance(node, PdsKeyValuePair) and node.key == "add_mod_only":
            mod_added_kvp = node
            break
    if mod_added_kvp:
        print(f"  PASS: MOD_ADDED node 'add_mod_only' found at ROOT.")
        # Path uses the node's actual key in the merged tree and its occurrence index relative to other nodes *of that key* at that level.
        # Since 'add_mod_only' is unique at the root, its path is just ['add_mod_only___0'].
        assert_node_value(merged_nodes, ['add_mod_only___0'], expected_value="mod_added", expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:MOD_ADDED")
    else:
        print(f"  FAIL: MOD_ADDED node 'add_mod_only' not found at ROOT.")


    # delete_new_only: VANILLA_DELETED -> Should be absent
    assert_node_absent(merged_nodes, ['delete_new_only___0'])

    # modify_mod_delete_new: CONFLICT_DELETION_VANILLA_DELETED_MOD_MODIFIED -> Mod's modified version applied as add
    # Should be present (find by key)
    mod_modified_added_kvp = None
    for node in root_container:
         if isinstance(node, PdsKeyValuePair) and node.key == "modify_mod_delete_new":
             mod_modified_added_kvp = node
             break
    if mod_modified_added_kvp:
         print(f"  PASS: CONFLICT_DELETION_VANILLA_DELETED_MOD_MODIFIED node 'modify_mod_delete_new' found at ROOT.")
         assert_node_value(merged_nodes, ['modify_mod_delete_new___0'], expected_value="modded_value", expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:CONFLICT_DELETION_VANILLA_DELETED_MOD_MODIFIED_MOD_MOD_APPLIED_AS_ADD")
    else:
        print(f"  FAIL: CONFLICT_DELETION_VANILLA_DELETED_MOD_MODIFIED node 'modify_mod_delete_new' not found at ROOT.")

    # delete_mod_keep_new: MOD_DELETED -> Should be absent
    assert_node_absent(merged_nodes, ['delete_mod_keep_new___0'])

    # keep_identical: IDENTICAL -> Should be present and identical
    assert_node_value(merged_nodes, ['keep_identical___0'], expected_value="unchanged", expected_type=PdsKeyValuePair, check_comment_substring=None) # No merge comment expected

    # Comment changes
    # Original comment -> CONFLICT_MODIFIED (Mod & Vanilla both modified) -> Mod Wins
    # The path is based on the original comment's content hash + index
    # Need to find the comment node by checking existing comment nodes
    original_comment_text = "# This is an original comment" # Need to know the original content to find it by hash
    comment_path_segment = f"__COMMENT_{hash(original_comment_text[:20])}___0"
    assert_node_value(merged_nodes, [comment_path_segment], expected_value="This is a modded comment", expected_type=PdsComment, check_comment_substring="SimMerge:CONFLICT_MODIFIED")

    # Blank line -> CONFLICT_MODIFIED (Mod made it comment, Vanilla made it comment) -> Mod Wins
    # Need to find the resulting comment node. It might have replaced the blank line node slot.
    # Its path segment is based on its *new* content hash.
    mod_comment_text = "# Mod replaced blank with comment"
    mod_comment_path_segment = f"__COMMENT_{hash(mod_comment_text[:20])}___0" # Assuming it's the first comment starting this way
    assert_node_value(merged_nodes, [mod_comment_path_segment], expected_value="# Mod replaced blank with comment", expected_type=PdsComment, check_comment_substring="SimMerge:CONFLICT_MODIFIED")


    # Section 3: Lists
    # simple_list_conflict: CONFLICT_MODIFIED list -> Should contain merged items
    # Old: { item_a item_b item_c }
    # Mod: { item_a item_mod item_d item_added_mod }
    # New: { item_a item_v item_e item_added_vanilla }
    # Expected merged: Item reconciling a/b/c modifications/deletions, and additions from both.
    # Reconciliation for list items is complex (position, value conflicts, additions).
    # Let's assert the *presence* of key items from both Mod and New, acknowledging order/specific conflicts might need refinement.
    list_path = ['simple_list_conflict___0']
    assert_node_value(merged_nodes, list_path, expected_type=PdsList, check_comment_substring="SimMerge:CONFLICT_MODIFIED")
    simple_list_node = _find_node_by_diff_path_in_tree(merged_nodes, list_path)
    if simple_list_node:
        merged_list_values = [v for v in simple_list_node.values if not isinstance(v, PdsNode)]
        print(f"  Debug: simple_list_conflict merged values: {merged_list_values}")
        assert "item_a" in merged_list_values # From Old, present in both modified lists
        assert "item_mod" in merged_list_values # Mod's modified 'item_b' or 'item_c'? This depends on diffing.
        assert "item_v" in merged_list_values # Vanilla's modified 'item_b' or 'item_c'?
        assert "item_d" in merged_list_values # Mod's added 'item_d'
        assert "item_e" in merged_list_values # Vanilla's added 'item_e'
        assert "item_added_mod" in merged_list_values # Mod's added 'item_added_mod'
        assert "item_added_vanilla" in merged_list_values # Vanilla's added 'item_added_vanilla'
        # Check list length heuristic? Difficult due to delete/replace/add combinations.
        print(f"  PASS: simple_list_conflict appears to contain key items from Mod and New.")
    else: print(f"  FAIL: simple_list_conflict node not found.")


    # multi_line_list: VANILLA_MODIFIED list -> Contains items from Old + Vanilla add. Mod add should be ignored?
    # Old: { item_1 item_2 }
    # Mod: { item_1 item_mod_added item_2 }
    # New: { item_1 item_2 item_vanilla_added }
    # Diff says VANILLA_MODIFIED. Merge logic comments the list node. Children diffs handle items.
    # item_mod_added: MOD_ADDED relative to Old list state -> Should be added
    # item_vanilla_added: VANILLA_ADDED relative to Old list state -> Should be added
    list_path = ['multi_line_list___0']
    assert_node_value(merged_nodes, list_path, expected_type=PdsList, check_comment_substring="SimMerge:VANILLA_MODIFIED")
    multi_list_node = _find_node_by_diff_path_in_tree(merged_nodes, list_path)
    if multi_list_node:
        merged_list_values = [v for v in multi_list_node.values if not isinstance(v, PdsNode)]
        print(f"  Debug: multi_line_list merged values: {merged_list_values}")
        assert "item_1" in merged_list_values
        assert "item_2" in merged_list_values
        assert "item_mod_added" in merged_list_values # Mod's addition should be present
        assert "item_vanilla_added" in merged_list_values # Vanilla's addition should be present
        print(f"  PASS: multi_line_list contains items from Old, Mod added, and Vanilla added.")
    else: print(f"  FAIL: multi_line_list node not found.")

    # List with anonymous blocks
    # First block (L88): MOD_MODIFIED (value change) -> Mod wins value
    block1_path = ['list_with_anon_blocks___0', '__BLOCK_L88___0']
    assert_node_value(merged_nodes, block1_path, expected_type=PdsBlock, check_comment_substring="SimMerge:MOD_MODIFIED")
    assert_node_value(merged_nodes, block1_path + ['block_item_1___0'], expected_value="mod_value_changed", expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:MOD_MODIFIED")

    # Second block (L89): VANILLA_MODIFIED (value change) -> Vanilla wins value
    block2_path = ['list_with_anon_blocks___0', '__BLOCK_L89___0']
    assert_node_value(merged_nodes, block2_path, expected_type=PdsBlock, check_comment_substring="SimMerge:VANILLA_MODIFIED")
    assert_node_value(merged_nodes, block2_path + ['block_item_2___0'], expected_value="vanilla_value_changed", expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:VANILLA_MODIFIED")

    # Mod added block: Should be present (find by key in list values)
    list_node = _find_node_by_diff_path_in_tree(merged_nodes, ['list_with_anon_blocks___0'])
    mod_added_anon_block = None
    if list_node and isinstance(list_node, PdsList):
        for item in list_node.values:
            if isinstance(item, PdsBlock) and item.find_child_by_key("block_item_3_mod"):
                 mod_added_anon_block = item
                 break
    if mod_added_anon_block:
         print(f"  PASS: Mod added anonymous block found in list.")
         assert_node_value([mod_added_anon_block], [_get_node_diff_key_for_find(mod_added_anon_block), 'block_item_3_mod___0'], expected_value="mod_added", expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:MOD_ADDED") # Check content and comment
    else: print(f"  FAIL: Mod added anonymous block not found in list.")

    # Vanilla added block: Should be present (find by key in list values)
    vanilla_added_anon_block = None
    if list_node and isinstance(list_node, PdsList):
         for item in list_node.values:
             if isinstance(item, PdsBlock) and item.find_child_by_key("block_item_3_vanilla"):
                  vanilla_added_anon_block = item
                  break
    if vanilla_added_anon_block:
         print(f"  PASS: Vanilla added anonymous block found in list.")
         assert_node_value([vanilla_added_anon_block], [_get_node_diff_key_for_find(vanilla_added_anon_block), 'block_item_3_vanilla___0'], expected_value="vanilla_added", expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:VANILLA_ADDED")
    else: print(f"  FAIL: Vanilla added anonymous block not found in list.")


    # Section 4: Blocks
    # simple_block_conflict: CONFLICT_MODIFIED Block -> Kept Vanilla structure, children merged
    block_path = ['simple_block_conflict___0']
    assert_node_value(merged_nodes, block_path, expected_type=PdsBlock, check_comment_substring="SimMerge:CONFLICT_MODIFIED_CONTENTS_MERGED_BY_CHILDREN")
    
    # Check children inside simple_block_conflict
    # child_kvp: CONFLICT_MODIFIED -> Mod wins
    assert_node_value(merged_nodes, block_path + ['child_kvp___0'], expected_value="mod_value_changed", expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:CONFLICT_MODIFIED_MOD_CHOSEN")
    # child_list: CONFLICT_MODIFIED (list content) -> Vanilla list structure kept, children merged
    list_child_path = block_path + ['child_list___0']
    assert_node_value(merged_nodes, list_child_path, expected_type=PdsList, check_comment_substring="SimMerge:CONFLICT_MODIFIED") # Comment on the list node itself
    list_child_node = _find_node_by_diff_path_in_tree(merged_nodes, list_child_path)
    if list_child_node:
         merged_list_values = [v for v in list_child_node.values if not isinstance(v, PdsNode)]
         print(f"  Debug: simple_block_conflict->child_list merged values: {merged_list_values}")
         assert "list_item_x" in merged_list_values
         assert "list_item_mod_y" in merged_list_values # Mod's modified y
         assert "list_item_y" not in merged_list_values # Original y replaced
         assert "list_item_vanilla_z" in merged_list_values # Vanilla added z here
         print(f"  PASS: simple_block_conflict->child_list contains merged items.")
    else: print(f"  FAIL: simple_block_conflict->child_list node not found.")

    # mod_added_child: MOD_ADDED -> Should be present
    assert_node_value(merged_nodes, block_path + ['mod_added_child___0'], expected_value=True, expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:MOD_ADDED")

    # Nested Blocks
    parent_path = ['parent_block___0']
    assert_node_value(merged_nodes, parent_path, expected_type=PdsBlock, check_comment_substring="SimMerge:VANILLA_MODIFIED") # Vanilla added comment
    
    # child_block_level1_mod: CONFLICT_MODIFIED Block -> Kept Vanilla structure, children merged
    level1_mod_path = parent_path + ['child_block_level1_mod___0']
    assert_node_value(merged_nodes, level1_mod_path, expected_type=PdsBlock, check_comment_substring="SimMerge:CONFLICT_MODIFIED_CONTENTS_MERGED_BY_CHILDREN") # Vanilla added comment + SimMerge comment
    
    # child_block_level2: CONFLICT_MODIFIED Block -> Kept Vanilla structure, children merged
    level2_path = level1_mod_path + ['child_block_level2___0']
    assert_node_value(merged_nodes, level2_path, expected_type=PdsBlock, check_comment_substring="SimMerge:CONFLICT_MODIFIED_CONTENTS_MERGED_BY_CHILDREN")

    # final_value (inside level2): CONFLICT_MODIFIED -> Mod wins
    assert_node_value(merged_nodes, level2_path + ['final_value___0'], expected_value="modded", expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:CONFLICT_MODIFIED_MOD_CHOSEN")

    # mod_added_kvp (inside level2): MOD_ADDED -> Should be present
    assert_node_value(merged_nodes, level2_path + ['mod_added_kvp___0'], expected_value=1, expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:MOD_ADDED")

    # vanilla_added_kvp (inside level2): VANILLA_ADDED -> Should be present
    assert_node_value(merged_nodes, level2_path + ['vanilla_added_kvp___0'], expected_value=2, expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:VANILLA_ADDED")

    # mod_added_at_level1: MOD_ADDED -> Should be present
    assert_node_value(merged_nodes, level1_mod_path + ['mod_added_at_level1___0'], expected_value="mod", expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:MOD_ADDED")

    # child_block_level1_new: VANILLA_ADDED -> Should be present (was deleted by Mod, kept by Vanilla)
    # Diff type: VANILLA_ADDED (o=Absent, m=Absent, n=Node)
    level1_new_path = parent_path + ['child_block_level1_new___0'] # Path based on original presence in Old/New
    assert_node_value(merged_nodes, level1_new_path, expected_type=PdsBlock, check_comment_substring="SimMerge:VANILLA_ADDED")
    assert_node_value(merged_nodes, level1_new_path + ['should_be_deleted_by_mod___0'], expected_value=True, expected_type=PdsKeyValuePair, check_comment_substring=None)


    # vanilla_added_block_level1: VANILLA_ADDED -> Should be present (find by key)
    parent_node = _find_node_by_diff_path_in_tree(merged_nodes, parent_path)
    vanilla_added_block = None
    if parent_node and isinstance(parent_node, PdsBlock):
         for child in parent_node.children:
             if isinstance(child, PdsBlock) and child.key == "vanilla_added_block_level1":
                 vanilla_added_block = child
                 break
    if vanilla_added_block:
         print(f"  PASS: Vanilla added block 'vanilla_added_block_level1' found.")
         assert_node_value([vanilla_added_block], [_get_node_diff_key_for_find(vanilla_added_block), 'child___0'], expected_value=True, expected_type=PdsKeyValuePair, check_comment_substring=None)
         assert_node_value([vanilla_added_block], [_get_node_diff_key_for_find(vanilla_added_block)], expected_type=PdsBlock, check_comment_substring="SimMerge:VANILLA_ADDED")
    else: print(f"  FAIL: Vanilla added block 'vanilla_added_block_level1' not found.")


    # modify_mod_delete_new_block: VANILLA_DELETED -> Block should be absent
    # Old: Block({...}); Mod: Block({...mod_added...}); New: Absent
    # Diff type: VANILLA_DELETED (o=Block, m=Block, n=Absent). Mod node is present in Mod diff, but Van deleted it.
    # Merge logic for VANILLA_DELETED notes it's absent.
    assert_node_absent(merged_nodes, ['modify_mod_delete_new_block___0'])

    # delete_mod_modify_new_block: CONFLICT_DELETION_VANILLA_DELETED_MOD_MODIFIED -> Mod's modified version applied as add
    # Old: Block({...}); Mod: Absent; New: Block({...van_added...})
    # Diff type: CONFLICT_DELETION_MOD_DELETED_VANILLA_MODIFIED (o=Block, m=Absent, n=Block)
    # Merge logic: Apply Mod's version as an add. Mod's version is ABSENT in Mod diff.
    # This looks like a case where the differ mapping might be slightly off.
    # Let's re-evaluate diff logic for `delete_mod_modify_new_block` / `delete_mod_modify_new_block`
    # delete_mod_modify_new_block (Old): key=delete_mod_modify_new_block, children={initial_child}
    # delete_mod_modify_new_block (Mod): Absent -> Old->Mod: DELETED
    # delete_mod_modify_new_block (New): key=delete_mod_modify_new_block, children={initial_child, new_child_van} -> Old->New: MODIFIED
    # Reconciliation: s_om='DELETED', s_on='MODIFIED' -> CONFLICT_DELETION_MOD_DELETED_VANILLA_MODIFIED
    # Merge logic for this type applies Mod's version as add. Mod's version is None. This would add None? No, it adds chg_obj.mod_node.copy() which is None.
    # This merge type should probably KEEP Vanilla's modified version.
    # Let's *assume* the fix should be to KEEP Vanilla's version for this conflict type.
    # Re-evaluate `_apply_single_change_to_sim_tree` for `CONFLICT_DELETION_MOD_DELETED_VANILLA_MODIFIED`
    # It currently applies Mod's node as add. Let's change it to KEEP Vanilla's node.
    # --> *Self-Correction*: The logic for `CONFLICT_DELETION_MOD_DELETED_VANILLA_MODIFIED` already keeps Vanilla's modified version.
    # The debug print `MOD_MOD_APPLIED_AS_ADD` was misleading.
    # Let's trust the current logic and assert the presence of the Vanilla modified block.

    block_path = ['delete_mod_modify_new_block___0']
    assert_node_value(merged_nodes, block_path, expected_type=PdsBlock, check_comment_substring="SimMerge:CONFLICT_DELETION_MOD_DELETED_VANILLA_MODIFIED_VANILLA_MOD_KEPT")
    assert_node_value(merged_nodes, block_path + ['initial_child___0'], expected_value=1, expected_type=PdsKeyValuePair, check_comment_substring=None)
    assert_node_value(merged_nodes, block_path + ['new_child_van___0'], expected_value=True, expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:VANILLA_ADDED") # Vanilla added child should be present


    # Section 5: Operator Conditions
    condition_path = ['condition_example___0']
    # Base block: CONFLICT_MODIFIED Block -> Kept Vanilla structure, children merged
    assert_node_value(merged_nodes, condition_path, expected_type=PdsBlock, check_comment_substring="SimMerge:CONFLICT_MODIFIED_CONTENTS_MERGED_BY_CHILDREN")

    # base_value condition: MOD_MODIFIED OpCond -> Mod wins
    assert_node_value(merged_nodes, condition_path + ['base_value___0'], expected_value=10, expected_type=PdsOperatorCondition, check_comment_substring="SimMerge:MOD_MODIFIED")
    op_node = _find_node_by_diff_path_in_tree(merged_nodes, condition_path + ['base_value___0'])
    if op_node:
        if op_node.operator == ">=": print("  PASS: condition_example->base_value has correct operator '>=' (Mod change).")
        else: print(f"  FAIL: condition_example->base_value has incorrect operator. Expected '>=', got '{op_node.operator}'.")
    else: print("  FAIL: condition_example->base_value node not found for operator check.")

    # always_true KVP: VANILLA_MODIFIED KVP -> Vanilla wins
    assert_node_value(merged_nodes, condition_path + ['always_true___0'], expected_value=False, expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:VANILLA_MODIFIED")

    # Comment within condition block
    # Old: # Comment CondItem1 Old
    # New: # Initial child comment CondItem1 Old # Vanilla modifies comment
    # Diff: VANILLA_MODIFIED comment
    comment_item_path = condition_path + ['__COMMENT_' + str(hash("# Comment CondItem1 Old"[:20])) + '___0']
    assert_node_value(merged_nodes, comment_item_path, expected_value="# Initial child comment CondItem1 Old # Vanilla modifies comment", expected_type=PdsComment, check_comment_substring="SimMerge:VANILLA_MODIFIED")


    # Section 6: Edge Cases & Mixes
    # kvp_to_block: MOD_MODIFIED -> Mod wins (KVP with Block value)
    kvp_to_block_path = ['kvp_to_block___0']
    assert_node_value(merged_nodes, kvp_to_block_path, expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:MOD_MODIFIED")
    kvp_node = _find_node_by_diff_path_in_tree(merged_nodes, kvp_to_block_path)
    if kvp_node and isinstance(kvp_node.value, PdsBlock):
        print(f"  PASS: kvp_to_block value is correctly a PdsBlock.")
        assert_node_value([kvp_node.value], [_get_node_diff_key_for_find(kvp_node.value), 'child_in_block___0'], expected_value="mod_value", expected_type=PdsKeyValuePair, check_comment_substring=None)
        assert_node_value([kvp_node.value], [_get_node_diff_key_for_find(kvp_node.value), 'another_child___0'], expected_value=True, expected_type=PdsKeyValuePair, check_comment_substring=None)
        # Path for the block value itself needs check on subsumption
        # The path for the block *value* is not kvp_to_block___0, it's the path *of the block itself* if it had one, or its generated path.
        # The subsumption should apply to paths *within* the block value. E.g., ['kvp_to_block___0', 'child_in_block___0'] should be skipped IF kvp_to_block___0's VALUE block was subsuming.
        # Let's check if the path of the KVP itself is subsumed.
        # Path for the KVP is ['kvp_to_block___0']. Its value is a Block.
        # The Mod_Modified rule for non-Blocks (which KVP is) *does* replace the node.
        # If the replacement node's value is a Block, should its path be subsumed? Yes.
        if tuple(kvp_to_block_path) in g_subsumed_mod_block_paths_for_simulation:
             print(f"  PASS: kvp_to_block path correctly subsumed.") # The KVP path is marked if its value is a block from Mod
        else:
             print(f"  FAIL: kvp_to_block path NOT correctly subsumed.")
    else:
        print(f"  FAIL: kvp_to_block node not found or value is not a PdsBlock. Got {type(kvp_node.value).__name__ if kvp_node and hasattr(kvp_node, 'value') else 'None'}.")


    # Block key changes: Old block deleted by Mod, New block added by Vanilla.
    # Old block (block_key_change_old): MOD_DELETED -> Absent
    assert_node_absent(merged_nodes, ['block_key_change_old___0'])
    # Mod block (block_key_change_mod): MOD_ADDED -> Present
    mod_key_block = None
    for node in merged_nodes:
        if isinstance(node, PdsBlock) and node.key == "block_key_change_mod":
            mod_key_block = node
            break
    if mod_key_block:
        print(f"  PASS: Mod added block 'block_key_change_mod' found.")
        assert_node_value([mod_key_block], [_get_node_diff_key_for_find(mod_key_block), 'value___0'], expected_value=1, expected_type=PdsKeyValuePair, check_comment_substring=None)
        assert_node_value([mod_key_block], [_get_node_diff_key_for_find(mod_key_block), 'mod_added___0'], expected_value=True, expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:MOD_ADDED")
        # Path of added block should be subsumed. Need its actual path.
        # Let's assert its presence and contents, checking subsumption is difficult without dynamic pathing.
        # Assume if the block is added and found, its children are also implicitly handled as mod additions relative to its structure.
    else: print(f"  FAIL: Mod added block 'block_key_change_mod' not found.")

    # New block (block_key_change_new): VANILLA_ADDED -> Present
    new_key_block_path = ['block_key_change_new___0']
    assert_node_value(merged_nodes, new_key_block_path, expected_type=PdsBlock, check_comment_substring="SimMerge:VANILLA_ADDED")
    assert_node_value(merged_nodes, new_key_block_path + ['value___0'], expected_value=1, expected_type=PdsKeyValuePair, check_comment_substring=None)
    assert_node_value(merged_nodes, new_key_block_path + ['vanilla_added___0'], expected_value=False, expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:VANILLA_ADDED")


    # empty_block: IDENTICAL -> Should be present and empty. Mod added comment.
    empty_block_path = ['empty_block___0']
    assert_node_value(merged_nodes, empty_block_path, expected_type=PdsBlock, check_comment_substring="SimMerge:MOD_MODIFIED") # Mod added comment to empty block -> MOD_MODIFIED
    empty_block_node = _find_node_by_diff_path_in_tree(merged_nodes, empty_block_path)
    if empty_block_node:
         if not empty_block_node.children: # Children list should be empty except for possible SimMerge comments
              # Check if there's exactly one child comment added by SimMerge
              sim_comments = [c for c in empty_block_node.children if isinstance(c, PdsComment) and "SimMerge:" in c.comment_text]
              if len(sim_comments) == 1:
                  print("  PASS: empty_block is present and contains only the SimMerge comment.")
              else:
                  print(f"  FAIL: empty_block has unexpected children or comments. Children count: {len(empty_block_node.children)}")
         else: print(f"  FAIL: empty_block is not empty. Contains children: {empty_block_node.children}")
    else: print("  FAIL: empty_block node not found.")


    # block_with_comments: CONFLICT_MODIFIED Block -> Kept Vanilla structure, comments and children merged
    comments_block_path = ['block_with_comments___0']
    assert_node_value(merged_nodes, comments_block_path, expected_type=PdsBlock, check_comment_substring="SimMerge:CONFLICT_MODIFIED_CONTENTS_MERGED_BY_CHILDREN")
    block_comments_node = _find_node_by_diff_path_in_tree(merged_nodes, comments_block_path)
    if block_comments_node:
        # Check for merged opening line comment
        if block_comments_node.comment_text_on_line and \
           "# Initial block comment Old" in block_comments_node.comment_text_on_line and \
           "Mod adds to block comment" in block_comments_node.comment_text_on_line and \
           "Vanilla adds to block comment" in block_comments_node.comment_text_on_line:
             print("  PASS: block_with_comments has merged opening line comment.")
        else:
             print(f"  FAIL: block_with_comments missing parts of opening line comment. Got: '{block_comments_node.comment_text_on_line}'")

        # Check for merged/added child comments
        # Initial child comment Old 1: CONFLICT_MODIFIED comment -> Mod wins
        comment1_path = comments_block_path + ['__COMMENT_' + str(hash("# Initial child comment Old 1"[:20])) + '___0']
        assert_node_value(merged_nodes, comment1_path, expected_value="# Initial child comment Old 1 # Mod modifies comment 1", expected_type=PdsComment, check_comment_substring="SimMerge:CONFLICT_MODIFIED")

        # Initial child comment Old 2: VANILLA_MODIFIED comment -> Vanilla wins
        comment2_path = comments_block_path + ['__COMMENT_' + str(hash("# Initial child comment Old 2"[:20])) + '___0']
        assert_node_value(merged_nodes, comment2_path, expected_value="# Initial child comment Old 2 # Vanilla modifies comment 2", expected_type=PdsComment, check_comment_substring="SimMerge:VANILLA_MODIFIED")

        # Mod adds child comment 2: MOD_ADDED comment -> Present
        # Need to find the comment node by its content
        mod_added_comment_text = "# Mod adds child comment 2"
        mod_added_comment_found = any(isinstance(c, PdsComment) and c.comment_text == mod_added_comment_text for c in block_comments_node.children)
        if mod_added_comment_found:
             # Find its path based on content hash and occurrence index
             mod_added_comment_path_seg = f'__COMMENT_{hash(mod_added_comment_text[:20])}___0'
             assert_node_value(merged_nodes, comments_block_path + [mod_added_comment_path_seg], expected_value=mod_added_comment_text, expected_type=PdsComment, check_comment_substring="SimMerge:MOD_ADDED")
        else: print("  FAIL: block_with_comments missing Mod added child comment.")

        # Check for added child KVP (Mod added one)
        assert_node_value(merged_nodes, comments_block_path + ['mod_child___0'], expected_value=1, expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:MOD_ADDED")

        # Original child KVP
        assert_node_value(merged_nodes, comments_block_path + ['child___0'], expected_value=True, expected_type=PdsKeyValuePair, check_comment_substring=None)

    else: print("  FAIL: block_with_comments node not found.")


    # complex_list_items: CONFLICT_MODIFIED list (because of item_value conflict) -> Vanilla list structure kept, children merged
    complex_list_path = ['complex_list_items___0']
    assert_node_value(merged_nodes, complex_list_path, expected_type=PdsList, check_comment_substring="SimMerge:CONFLICT_MODIFIED") # Should be CONFLICT_MODIFIED due to item_value conflict
    complex_list_node = _find_node_by_diff_path_in_tree(merged_nodes, complex_list_path)

    if complex_list_node and isinstance(complex_list_node, PdsList):
        print("  PASS: complex_list_items node found.")
        # Check first anonymous block (L188): CONFLICT_MODIFIED (x mod, y van added) -> Block kept, children merged
        first_anon_block_path_seg = '__BLOCK_L188___0'
        assert_node_value(merged_nodes, complex_list_path + [first_anon_block_path_seg], expected_type=PdsBlock, check_comment_substring="SimMerge:CONFLICT_MODIFIED")
        first_anon_block = _find_node_by_diff_path_in_tree(merged_nodes, complex_list_path + [first_anon_block_path_seg])
        if first_anon_block:
             print("  PASS: complex_list_items first anonymous block found.")
             assert_node_value([first_anon_block], [_get_node_diff_key_for_find(first_anon_block), 'complex_block___0', 'x___0'], expected_value="mod_changed", expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:MOD_MODIFIED")
             assert_node_value([first_anon_block], [_get_node_diff_key_for_find(first_anon_block), 'complex_block___0', 'y___0'], expected_value=2, expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:VANILLA_ADDED")
        else: print("  FAIL: complex_list_items first anonymous block not found.")

        # item_value (L189): MOD_DELETED vs VANILLA_MODIFIED -> CONFLICT_DELETION_MOD_DELETED_VANILLA_MODIFIED -> Vanilla modified kept
        # This is a bare string item in the list. Diff path will be item_value___0.
        # The value should be "item_value_vanilla_modified". How to check comment on it? Current AST doesn't support comments on bare list items.
        value_item_found = False
        expected_value = "item_value_vanilla_modified"
        for item in complex_list_node.values:
             if not isinstance(item, PdsNode) and item == expected_value:
                 value_item_found = True
                 break
        if value_item_found:
             print(f"  PASS: complex_list_items '{expected_value}' found (Vanilla modified kept).")
             # Cannot assert comment on bare string/number/bool list items
        else: print(f"  FAIL: complex_list_items '{expected_value}' not found.")

        # "quoted string" (L190): MOD_MODIFIED -> Mod wins
        quoted_item_found = False
        expected_value = "mod string"
        for item in complex_list_node.values:
             if not isinstance(item, PdsNode) and item == expected_value:
                 quoted_item_found = True
                 break
        if quoted_item_found:
             print(f"  PASS: complex_list_items '{expected_value}' found (Mod modified kept).")
             # Cannot assert comment on bare string/number/bool list items
        else: print(f"  FAIL: complex_list_items '{expected_value}' not found.")


        # mod_added_item = "mod_added": MOD_ADDED -> Should be present (This is a KVP in a List)
        mod_added_item_found = False
        for item in complex_list_node.values:
             if isinstance(item, PdsKeyValuePair) and item.key == "mod_added_item":
                 mod_added_item_found = True
                 assert item.value == "mod_added"
                 # Path for KVP in list: ['complex_list_items___0', 'mod_added_item___0'] # Assuming it's the first with this key
                 assert_node_value(merged_nodes, complex_list_path + ['mod_added_item___0'], expected_value="mod_added", expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:MOD_ADDED")
                 break
        if mod_added_item_found: print("  PASS: complex_list_items 'mod_added_item' KVP found.")
        else: print("  FAIL: complex_list_items 'mod_added_item' KVP not found.")

    else: print("  FAIL: complex_list_items node not found.")


    # --- Section 7: Random List / Weighted Blocks Stress Test ---
    random_list_path = ['random_list_benchmark___0']
    # Random List block itself: CONFLICT_MODIFIED (due to changes within) -> Kept Vanilla structure
    assert_node_value(merged_nodes, random_list_path, expected_type=PdsList, check_comment_substring="SimMerge:CONFLICT_MODIFIED")

    # 10 = {...}: CONFLICT_MODIFIED Block (Mod modified, Vanilla modified+deleted)
    # Old: 10={a=yes, b=1}; Mod: 10={a="mod_value", b=99, mod_added_10=yes}; New: 10={b=99} (effectively deleted a)
    # Diff: Old->Mod: 10={...} MODIFIED. Old->New: 10={...} MODIFIED (a deleted, b modified).
    # Reconciliation: MODIFIED vs MODIFIED -> CONFLICT_MODIFIED Block.
    # Merge Logic: Keep Vanilla Block structure, children merged.
    # This case means the block with key '10' exists in all three versions.
    weight_10_path = random_list_path + ['10___0']
    assert_node_value(merged_nodes, weight_10_path, expected_type=PdsBlock, check_comment_substring="SimMerge:CONFLICT_MODIFIED_CONTENTS_MERGED_BY_CHILDREN")
    
    weight_10_block = _find_node_by_diff_path_in_tree(merged_nodes, weight_10_path)
    if weight_10_block:
         print("  PASS: Weighted block 10 found.")
         # item_a: MOD_MODIFIED vs VANILLA_DELETED -> CONFLICT_DELETION_VANILLA_DELETED_MOD_MODIFIED -> Vanilla deleted kept (absent)
         assert_node_absent(merged_nodes, weight_10_path + ['item_a___0']) # Item a should be absent

         # item_b: CONVERGED_MODIFICATION -> Mod and Vanilla both changed to 99
         assert_node_value(merged_nodes, weight_10_path + ['item_b___0'], expected_value=99, expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:CONVERGED_MODIFICATION")

         # mod_added_10: MOD_ADDED -> Present
         assert_node_value(merged_nodes, weight_10_path + ['mod_added_10___0'], expected_value=True, expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:MOD_ADDED")
    else: print("  FAIL: Weighted block 10 not found.")


    # 20 = {...}: MOD_MODIFIED Block vs VANILLA_DELETED -> CONFLICT_DELETION_MOD_DELETED_VANILLA_MODIFIED
    # Old: 20={item_c="original"}; Mod: 20={item_c="modded", mod_added_20=no}; New: Absent
    # Merge Logic: Apply Mod's modified version as an add. Mod's version should be present.
    # The path in the merged tree will likely be '20___INSERTED...'. Finding it by key is safer.
    weight_20_block = None
    random_list_node = _find_node_by_diff_path_in_tree(merged_nodes, random_list_path)
    if random_list_node and isinstance(random_list_node, PdsList):
        for item in random_list_node.values:
            if isinstance(item, PdsBlock) and item.key == 20:
                weight_20_block = item
                break
    if weight_20_block:
        print("  PASS: Weighted block 20 found (applied as add).")
        assert_node_value([weight_20_block], [_get_node_diff_key_for_find(weight_20_block), 'item_c___0'], expected_value="modded", expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:CONFLICT_DELETION_MOD_DELETED_VANILLA_MODIFIED_MOD_MOD_APPLIED_AS_ADD")
        assert_node_value([weight_20_block], [_get_node_diff_key_for_find(weight_20_block), 'mod_added_20___0'], expected_value=False, expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:MOD_ADDED")
    else: print("  FAIL: Weighted block 20 not found.")


    # 30 = {...}: MOD_MODIFIED Block vs IDENTICAL -> MOD_MODIFIED Block
    # Old: 30={item_d=no}; Mod: 30={item_d=yes}; New: 30={item_d=no}
    # Diff: Old->Mod: 30={...} MODIFIED. Old->New: 30={...} IDENTICAL.
    # Reconciliation: MODIFIED vs IDENTICAL -> MOD_MODIFIED Block
    # Merge Logic: Keep Vanilla Block structure, children merged.
    # item_d: MOD_MODIFIED vs IDENTICAL -> MOD_MODIFIED -> Mod wins value
    weight_30_path = random_list_path + ['30___0']
    assert_node_value(merged_nodes, weight_30_path, expected_type=PdsBlock, check_comment_substring="SimMerge:MOD_MODIFIED_CONTENTS_MERGED_BY_CHILDREN")
    assert_node_value(merged_nodes, weight_30_path + ['item_d___0'], expected_value=True, expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:MOD_MODIFIED")


    # 40 = {...}: IDENTICAL vs VANILLA_MODIFIED Block -> VANILLA_MODIFIED Block
    # Old: 40={item_e=1.5}; Mod: 40={item_e=1.5}; New: 40={item_e=2.5, vanilla_added_40=no}
    # Diff: Old->Mod: 40={...} IDENTICAL. Old->New: 40={...} MODIFIED.
    # Reconciliation: IDENTICAL vs MODIFIED -> VANILLA_MODIFIED Block
    # Merge Logic: Keep Vanilla Block structure, children merged.
    # item_e: IDENTICAL vs VANILLA_MODIFIED -> VANILLA_MODIFIED -> Vanilla wins value
    weight_40_path = random_list_path + ['40___0']
    assert_node_value(merged_nodes, weight_40_path, expected_type=PdsBlock, check_comment_substring="SimMerge:VANILLA_MODIFIED_CONTENTS_MERGED_BY_CHILDREN")
    assert_node_value(merged_nodes, weight_40_path + ['item_e___0'], expected_value=2.5, expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:VANILLA_MODIFIED")
    # vanilla_added_40: VANILLA_ADDED -> Present
    assert_node_value(merged_nodes, weight_40_path + ['vanilla_added_40___0'], expected_value=False, expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:VANILLA_ADDED")


    # 50 = {...}: IDENTICAL -> Should be present and identical. Both Mod and Vanilla added comments.
    # Old: 50={item_f="keep"}
    # Mod: 50={item_f="keep" # Mod adds comment} -> MOD_MODIFIED (due to comment)
    # New: 50={item_f="keep" # Vanilla adds comment} -> MODIFIED (due to comment)
    # Reconciliation: MODIFIED vs MODIFIED -> CONFLICT_MODIFIED Block
    # Merge Logic: Keep Vanilla Block, children merged. Comments should be merged.
    weight_50_path = random_list_path + ['50___0']
    assert_node_value(merged_nodes, weight_50_path, expected_type=PdsBlock, check_comment_substring="SimMerge:CONFLICT_MODIFIED_CONTENTS_MERGED_BY_CHILDREN")
    assert_node_value(merged_nodes, weight_50_path + ['item_f___0'], expected_value="keep", expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:CONVERGED_MODIFICATION") # Both kept value identical

    weight_50_block = _find_node_by_diff_path_in_tree(merged_nodes, weight_50_path)
    if weight_50_block:
         print("  PASS: Weighted block 50 found.")
         # Check comments inside 50 block
         # Mod adds comment, Vanilla adds comment -> Both should be present as child comments?
         # Let's check for the comment texts.
         mod_comment_found = any(isinstance(c, PdsComment) and "Mod adds comment to 50 block" in c.comment_text for c in weight_50_block.children)
         van_comment_found = any(isinstance(c, PdsComment) and "Vanilla adds comment to 50 block" in c.children for c in weight_50_block.children) # Comments are on child KVP? No, on block itself.
         # Wait, comments on blocks are added as children by _sim_add_comment_to_node for Blocks.
         # But the comments in the benchmark file are *line comments*.
         # The diffing should pick up modification *of the line comment on the block definition line*.
         # Old: 50 = { # Comment 50 Old }
         # Mod: 50 = { # Comment 50 Old # Mod adds comment to 50 block } -> Line comment modified
         # New: 50 = { # Comment 50 Old # Vanilla adds comment to 50 block } -> Line comment modified
         # This is a CONFLICT_MODIFIED KVP (the 50={...} itself).
         # My previous logic for CONFLICT_MODIFIED on Blocks should handle this - the Block Node gets commented.
         # Line comments on the block's own line are part of the Block node's attributes.
         # Let's re-read the benchmark... ah, the comments ARE within the block body, *not* on the line with "50 = {".
         # Mod added comment inside 50 block. Vanilla added comment inside 50 block.
         mod_comment_text = "# Mod adds comment to 50 block"
         van_comment_text = "# Vanilla adds comment to 50 block"
         mod_comment_found = any(isinstance(c, PdsComment) and c.comment_text == mod_comment_text for c in weight_50_block.children)
         van_comment_found = any(isinstance(c, PdsComment) and c.comment_text == van_comment_text for c in weight_50_block.children)

         if mod_comment_found and van_comment_found:
             print("  PASS: Weighted block 50 contains comments added by both Mod and Vanilla.")
             # We'd need to check the merge comments on these comments nodes themselves if we wanted full rigor.
         else: print("  FAIL: Weighted block 50 missing comments added by Mod and/or Vanilla.")

    else: print("  FAIL: Weighted block 50 not found.")


    # 60 = {...}: MOD_ADDED -> Present
    weight_60_block = None
    if random_list_node and isinstance(random_list_node, PdsList):
        for item in random_list_node.values:
            if isinstance(item, PdsBlock) and item.key == 60:
                weight_60_block = item
                break
    if weight_60_block:
        print("  PASS: Weighted block 60 found (Mod added).")
        assert_node_value([weight_60_block], [_get_node_diff_key_for_find(weight_60_block), 'item_g___0'], expected_value="mod_added_60", expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:MOD_ADDED")
        assert_node_value([weight_60_block], [_get_node_diff_key_for_find(weight_60_block), 'nested_mod_60___0'], expected_type=PdsBlock, check_comment_substring="SimMerge:MOD_ADDED")
    else: print("  FAIL: Weighted block 60 not found.")


    # 70 = {...}: VANILLA_ADDED (similar to Old 20={...}) -> Present
    # Old 20 was DELETED by New. New ADDED 70.
    # Diff will show 20___0 as DELETED in New. 70___INSERTED... as ADDED in New.
    # This is a Vanilla rename/replace situation.
    weight_70_block = None
    if random_list_node and isinstance(random_list_node, PdsList):
        for item in random_list_node.values:
            if isinstance(item, PdsBlock) and item.key == 70:
                weight_70_block = item
                break
    if weight_70_block:
        print("  PASS: Weighted block 70 found (Vanilla added, effectively renamed 20).")
        # Check content from New version
        assert_node_value([weight_70_block], [_get_node_diff_key_for_find(weight_70_block), 'item_c___0'], expected_value="original", expected_type=PdsKeyValuePair, check_comment_substring=None) # Content from Old 20, not modded
        assert_node_value([weight_70_block], [_get_node_diff_key_for_find(weight_70_block), 'new_added_70___0'], expected_value=True, expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:VANILLA_ADDED")
        assert_node_value([weight_70_block], [_get_node_diff_key_for_find(weight_70_block)], expected_type=PdsBlock, check_comment_substring="SimMerge:VANILLA_ADDED") # Comment on block
    else: print("  FAIL: Weighted block 70 not found.")

    # 80 = {...}: VANILLA_ADDED -> Present
    weight_80_block = None
    if random_list_node and isinstance(random_list_node, PdsList):
        for item in random_list_node.values:
            if isinstance(item, PdsBlock) and item.key == 80:
                weight_80_block = item
                break
    if weight_80_block:
        print("  PASS: Weighted block 80 found (Vanilla added).")
        assert_node_value([weight_80_block], [_get_node_diff_key_for_find(weight_80_block), 'item_h___0'], expected_value=5, expected_type=PdsKeyValuePair, check_comment_substring="SimMerge:VANILLA_ADDED")
    else: print("  FAIL: Weighted block 80 not found.")


    # 20 = {...}: DELETED in New, MODIFIED in Mod. -> Conflict
    # Diff type: CONFLICT_DELETION_MOD_DELETED_VANILLA_MODIFIED (o=20 block, m=20 block modded, n=Absent)
    # Merge Logic: Apply Mod's modified version as an add? Or is 20___0 now re-evaluated against the tree state?
    # When the merge hits the change for 20___0, target_node_in_sim is None (deleted by Vanilla).
    # It's CONFLICT_DELETION_MOD_DELETED_VANILLA_MODIFIED. Logic applies Mod's version as add.
    # Need to check if Mod's modified version of the 20 block (key=20) was added *somewhere*.
    # This is the tricky case identified earlier. The assertion for finding block 20 above checks if it exists *by key*.
    # If it was added with key 20, it should pass the previous assertion.

    # Check if Vanilla's old 20 block is absent by its original path segment, despite Mod's changes.
    # The vanilla tree copy (sim_tree starts with this) did *not* contain the 20 block.
    # The merge logic for CONFLICT_DELETION_MOD_DELETED_VANILLA_MODIFIED *adds* Mod's version.
    # So the 20 block *should* be present in the final merged tree.
    # The assertion for `weight_20_block` above already covers its presence and content if it's found by key 20.


    # 70 = {...} Added by New.
    # The diff might not see this as related to Old 20. It's just a VANILLA_ADDED.
    # The assertion for `weight_70_block` above checks its presence and content.

    print("\n--- Specific Benchmark Assertions Complete ---")


# --- Write Benchmark Files (Execute this once) ---
def write_benchmark_files():
    print("Writing benchmark files...")
    # Using the multi-line strings defined above
    write_to_file(BENCHMARK_OLD, """# benchmark_test_old.txt
# Base file for PDS Differ benchmark.

# --- Section 1: Primitive Value Changes & Conflicts ---
primitive_string = "initial_string"
primitive_number = 100
primitive_bool = yes
primitive_float = 1.0

# --- Section 2: Additions, Deletions, Unchanged ---
# This KVP will be deleted in Mod and Modified in New
delete_mod_modify_new = { # Comment Old
	value = 1
	keep_me = yes # Keep this child Old
}

# This KVP will be added in Mod only
# (Absent)

# This KVP will be deleted in New only
delete_new_only = 123

# This KVP will be modified in Mod, deleted in New
modify_mod_delete_new = "original"

# This KVP will be deleted in Mod, kept in New
delete_mod_keep_new = "original"

# This KVP will be kept identical in both
keep_identical = "unchanged" # Comment Identical

# This comment will be modified in Mod and New
# This is an original comment

# This blank line will be changed to a comment in Mod and New
# 

# --- Section 3: Lists ---
# Simple list, items modified/added/deleted in Mod and New
simple_list_conflict = { item_a item_b item_c } # Comment List Old

# Multi-line list, adds in Mod and New
multi_line_list = { # Comment MultiList Old
	item_1
	item_2
}

# List with anonymous blocks - children changed
list_with_anon_blocks = { # Comment AnonList Old
	{ block_item_1 = value_1 } # L88 - Comment Anon1 Old
	{ block_item_2 = value_2 } # L89 - Comment Anon2 Old
}

# --- Section 4: Blocks ---
# Simple block, children modified in Mod and New
simple_block_conflict = { # Comment Block Old
	child_kvp = base_value # Comment ChildKVP Old
	child_list = { list_item_x list_item_y } # Comment ChildList Old
}

# Nested blocks, changes at different levels
parent_block = { # Comment Parent Old
	child_block_level1_mod = { # Comment L1Mod Old
		child_block_level2 = { # Comment L2 Old
			final_value = "base" # Modified in Mod and New # Comment Final Old
			# Vanilla adds child here
		}
		# Mod adds KVP here
	}
	# Mod deletes child_block_level1_new below
	child_block_level1_new = { # Comment L1New Old
		should_be_deleted_by_mod = yes
	}
	# Vanilla adds block here
}

# Block modified by Mod (adds KVP), deleted by Vanilla
modify_mod_delete_new_block = { # Comment ModDelBlock Old
	initial_child = 1 # Comment InitChild Old
	# Mod adds new_child_mod here
}

# Block deleted by Mod, modified by Vanilla (adds KVP)
delete_mod_modify_new_block = { # Comment DelModBlock Old
	initial_child = 1 # Comment InitChild Old
	# Vanilla adds new_child_van here
}

# --- Section 5: Operator Conditions ---
condition_example = { # Comment CondBlock Old
	base_value > 10 # Comment CondItem1 Old
	always_true = yes # Vanilla changes this KVP # Comment CondItem2 Old
}

# --- Section 6: Edge Cases & Mixes ---
# KVP value changes from primitive to block (in Mod)
kvp_to_block = "its_a_string_originally" # Comment KVPToBlock Old

# Block key changes (simulate by delete+add)
block_key_change_old = { # Comment KeyChange Old
	value = 1
}

# Empty block initially
empty_block = { # Comment Empty Old
}

# Block with comments on brace line and child comments
block_with_comments = { # Initial block comment Old
	# Initial child comment Old 1
	# Initial child comment Old 2
	child = yes # Comment Child Old
}

# Orphaned nodes (should ideally not happen in valid PDS, but parser might produce)
# Orphan = "value" # This would be a top-level KVP

# More complex list item modifications
complex_list_items = { # Comment ComplexList Old
	{ complex_block = { x = 1 } } # Mod changes x, New adds y # L188 - Comment ComplexAnon Old
	item_value # Mod deletes, New modifies # L189
	"quoted string" # Mod modifies # L190
}

# --- Section 7: Random List / Weighted Blocks Stress Test ---
# This section tests lists where keys are numbers (weights) and values are blocks.
# Changes to weights AND internal block content are included.
random_list_benchmark = { # Comment RandomList Old
	10 = { # L198 - Weight 10 Old
		# Comment 10 Block Old
		item_a = yes # Modified in Mod, Deleted in New
		item_b = 1 # Modified in Mod and New
	}
	20 = { # L203 - Weight 20 Old - Deleted in New, Modified in Mod
		item_c = "original" # Modified in Mod
	}
	30 = { # L206 - Weight 30 Old - Kept in New, Modified in Mod
		item_d = no # Modified in Mod
	}
	40 = { # L209 - Weight 40 Old - Modified in New, Kept in Mod
		item_e = 1.5 # Modified in New
	}
	50 = { # L212 - Weight 50 Old - Kept Identical
		item_f = "keep" # Kept Identical
		# Mod adds comment to 50 block
		# Vanilla adds comment to 50 block
	}
	# Mod adds 60={...}
	# New adds 70={...} (similar to Old 20={...})
	# New adds 80={...}
}
""");

    write_to_file(BENCHMARK_MOD, """# benchmark_test_mod.txt
# Modded version for PDS Differ benchmark.

# --- Section 1: Primitive Value Changes & Conflicts ---
primitive_string = "mod_string_changed"
primitive_number = 999.9
primitive_bool = no
primitive_float = 1.0 # Kept by Mod

# --- Section 2: Additions, Deletions, Unchanged ---
# This KVP will be deleted in Mod and Modified in New
# (Absent)

# This KVP will be added in Mod only
add_mod_only = "mod_added" # Comment Mod Added

# This KVP will be deleted in New only
delete_new_only = 123 # Kept identical in Mod

# This KVP will be modified in Mod, deleted in New
modify_mod_delete_new = "modded_value" # Comment Mod Modified

# This KVP will be deleted in Mod, kept in New
# (Absent)

# This KVP will be kept identical in both
keep_identical = "unchanged" # Comment Identical

# This comment will be modified in Mod and New
# This is a modded comment

# This blank line will be changed to a comment in Mod and New
# # Mod replaced blank with comment

# --- Section 3: Lists ---
# Simple list, items modified/added/deleted in Mod and New
simple_list_conflict = { item_a item_mod item_d item_added_mod } # Mod changed b, c, added d, added item_added_mod

# Multi-line list, adds in Mod and New
multi_line_list = {
	item_1
	item_mod_added # Mod adds here
	item_2
}

# List with anonymous blocks - children changed
list_with_anon_blocks = {
	{ block_item_1 = mod_value_changed } # Mod changed value inside
	{ block_item_2 = value_2 }
	{ block_item_3_mod = mod_added } # Mod adds new anon block
}

# --- Section 4: Blocks ---
# Simple block, children modified in Mod and New
simple_block_conflict = { # Mod adds comment to block
	# Mod adds child comment
	child_kvp = mod_value_changed # Mod changes value
	child_list = { list_item_x list_item_mod_y } # Mod changed y
	mod_added_child = yes # Mod adds new child
}

# Nested blocks, changes at different levels
parent_block = {
	child_block_level1_mod = {
		child_block_level2 = {
			final_value = "modded" # Modified in Mod
			mod_added_kvp = 1 # Mod adds child here
		}
		mod_added_at_level1 = "mod" # Mod adds KVP here
	}
	# Mod deletes child_block_level1_new below
	# (Absent)
	# Vanilla adds block here (Absent in Mod)
}

# Block modified by Mod (adds KVP), deleted by Vanilla
modify_mod_delete_new_block = {
	initial_child = 1
	new_child_mod = yes # Mod adds new_child_mod here
}

# Block deleted by Mod, modified by Vanilla (adds KVP)
# (Absent)

# --- Section 5: Operator Conditions ---
condition_example = { # Mod adds block comment
	# Comment CondBlock Old # Mod adds to block comment line
	base_value >= 10 # Mod changes operator and value
	always_true = yes # Kept by Mod
}

# --- Section 6: Edge Cases & Mixes ---
# KVP value changes from primitive to block (in Mod)
kvp_to_block = { # Mod changes value to a block
	child_in_block = mod_value
	another_child = yes
} # Comment KVPToBlock Mod

# Block key changes (simulate by delete+add)
block_key_change_mod = { # Mod changes key
	value = 1
	mod_added = yes # Mod adds a child
} # Comment KeyChange Mod

# Empty block initially
empty_block = { # Mod adds comment to empty block
}

# Block with comments on brace line and child comments
block_with_comments = { # Initial block comment Old # Mod adds to block comment
	# Initial child comment Old 1 # Mod modifies comment 1
	# Initial child comment Old 2
	# Mod adds child comment 2
	child = yes
	mod_child = 1 # Mod adds child
}

# Orphaned nodes (Absent in Mod)
# Orphan = "value"

# More complex list item modifications
complex_list_items = {
	{ complex_block = { x = mod_changed } } # Mod changes x
	# item_value deleted by Mod
	"mod string" # Mod modifies quoted string
	mod_added_item = "mod_added" # Mod adds KVP to list
}

# --- Section 7: Random List / Weighted Blocks Stress Test ---
random_list_benchmark = { # Comment RandomList Old # Mod adds to block comment line
	10 = { # L198 - Weight 10 Old # Mod adds to block comment line
		# Comment 10 Block Old
		item_a = mod_value # Modified in Mod, Deleted in New
		item_b = 99 # Modified in Mod and New
		mod_added_10 = yes # Mod adds to 10 block
	}
	20 = { # L203 - Weight 20 Old - Deleted in New, Modified in Mod # Mod adds to block comment line
		item_c = "modded" # Modified in Mod
		mod_added_20 = no # Mod adds to 20 block
	}
	30 = { # L206 - Weight 30 Old - Kept in New, Modified in Mod
		item_d = yes # Modified in Mod
	}
	40 = { # L209 - Weight 40 Old - Modified in New, Kept in Mod
		item_e = 1.5 # Kept in Mod
	}
	50 = { # L212 - Weight 50 Old - Kept Identical
		item_f = "keep" # Kept Identical
		# Mod adds comment to 50 block
	}
	60 = { # Mod adds 60={...}
		item_g = "mod_added_60" # Mod adds to 60 block
		nested_mod_60 = { mod_val = 1 } # Mod adds nested block
	}
	# New adds 70={...} (similar to Old 20={...})
	# New adds 80={...}
}
""");

    write_to_file(BENCHMARK_NEW, """# benchmark_test_new.txt
# New Vanilla version for PDS Differ benchmark.

# --- Section 1: Primitive Value Changes & Conflicts ---
primitive_string = "vanilla_string_changed"
primitive_number = 100
primitive_bool = yes
primitive_float = 2.0 # Vanilla changes float

# --- Section 2: Additions, Deletions, Unchanged ---
# This KVP will be deleted in Mod and Modified in New
delete_mod_modify_new = { # Comment New
	value = 2 # Vanilla modifies value
	keep_me = yes
	vanilla_added_child = no # Vanilla adds child
}

# This KVP will be added in Mod only (Absent in New)
# (Absent)

# This KVP will be deleted in New only
# (Absent)

# This KVP will be modified in Mod, deleted in New
# (Absent)

# This KVP will be deleted in Mod, kept in New
delete_mod_keep_new = "original"

# This KVP will be kept identical in both
keep_identical = "unchanged" # Comment Identical

# This comment will be modified in Mod and New
# This is an original comment # Vanilla adds to comment

# This blank line will be changed to a comment in Mod and New
# # Vanilla replaced blank with comment

# --- Section 3: Lists ---
# Simple list, items modified/added/deleted in Mod and New
simple_list_conflict = { item_a item_v item_e item_added_vanilla } # Vanilla changed b, c, added e, added item_added_vanilla

# Multi-line list, adds in Mod and New
multi_line_list = {
	item_1
	item_2
	item_vanilla_added # Vanilla adds here
}

# List with anonymous blocks - children changed
list_with_anon_blocks = {
	{ block_item_1 = value_1 } 
	{ block_item_2 = vanilla_value_changed } # Vanilla changed value inside
	{ block_item_3_vanilla = vanilla_added } # Vanilla adds new anon block
}

# --- Section 4: Blocks ---
# Simple block, children modified in Mod and New
simple_block_conflict = { # Vanilla adds comment to block
	# Initial child comment Old 1 # Vanilla modifies comment 1
	child_kvp = vanilla_value_changed # Vanilla changes value
	child_list = { list_item_x list_item_y # Kept original
		list_item_vanilla_z # Vanilla adds item to list
	}
}

# Nested blocks, changes at different levels
parent_block = { # Vanilla adds block comment
	child_block_level1_mod = { # Vanilla adds comment
		child_block_level2 = {
			final_value = "vanilla" # Modified in New
			vanilla_added_kvp = 2 # Vanilla adds child here
		}
	}
	child_block_level1_new = { # Kept by Vanilla (not deleted)
		should_be_deleted_by_mod = yes
	}
	vanilla_added_block_level1 = { # Vanilla adds block here
		child = yes
	}
}

# Block modified by Mod (adds KVP), deleted by Vanilla
# (Absent)

# Block deleted by Mod, modified by Vanilla (adds KVP)
delete_mod_modify_new_block = { # Comment DelModBlock New
	initial_child = 1
	new_child_van = yes # Vanilla adds new_child_van here
}

# --- Section 5: Operator Conditions ---
condition_example = { # Comment CondBlock Old # Vanilla adds to block comment line
	# Initial child comment CondItem1 Old # Vanilla modifies comment
	base_value > 10 # Kept by Vanilla
	always_true = no # Vanilla changes this KVP's value
}

# --- Section 6: Edge Cases & Mixes ---
# KVP value changes from primitive to block (Absent in New)
kvp_to_block = "its_a_string_originally" # Kept by Vanilla as string

# Block key changes (simulate by delete+add)
block_key_change_old = { # Deleted by Vanilla
	value = 1
}
block_key_change_new = { # Vanilla adds block with new key
	value = 1
	vanilla_added = no # Vanilla adds a child
} # Comment KeyChange New

# Empty block initially
empty_block = {
} # Kept identical in New

# Block with comments on brace line and child comments
block_with_comments = { # Initial block comment Old # Vanilla adds to block comment
	# Initial child comment Old 1 # Vanilla modifies comment 1
	# Initial child comment Old 2 # Vanilla modifies comment 2
	child = yes
}

# Orphaned nodes (Absent in New)
# Orphan = "value"

# More complex list item modifications
complex_list_items = {
	{ complex_block = { x = 1 y = 2 } } # New adds y
	item_value_vanilla_modified # New modifies
	"quoted string" # Kept identical
}

# --- Section 7: Random List / Weighted Blocks Stress Test ---
random_list_benchmark = { # Comment RandomList Old # Vanilla adds to block comment line
	10 = { # L198 - Weight 10 Old - Modified in Mod, Deleted in New
		# Comment 10 Block Old # Vanilla modifies comment
		item_b = 99 # Modified in Mod and New
	}
	# 20 = {...} Deleted in New, Modified in Mod
	30 = { # L206 - Weight 30 Old - Kept in New, Modified in Mod
		item_d = no # Kept in New
	}
	40 = { # L209 - Weight 40 Old - Modified in New, Kept in Mod
		item_e = 2.5 # Modified in New
		vanilla_added_40 = no # Vanilla adds to 40 block
	}
	50 = { # L212 - Weight 50 Old - Kept Identical
		item_f = "keep" # Kept Identical
		# Mod adds comment to 50 block
		# Vanilla adds comment to 50 block
	}
	# Mod adds 60={...}
	70 = { # New adds 70={...} (similar to Old 20={...})
		item_c = "original" # Content from Old 20
		new_added_70 = yes # Vanilla adds to 70 block
	}
	80 = { # New adds 80={...}
		item_h = 5 # Vanilla adds to 80 block
	}
}
""");
    print("Benchmark files written.")


# --- Run Tests ---
print("Writing and running benchmark tests.")
write_benchmark_files() # Create the benchmark files

# Comment out real file tests for focused benchmark
# run_and_print_diff("00_tribal_innovations (real files)", INNOVATIONS_OLD_VANILLA_PATH, INNOVATIONS_MOD_PATH, INNOVATIONS_NEW_VANILLA_PATH)
# run_and_print_diff("00_traits (real files)", TRAITS_OLD_VANILLA_PATH, TRAITS_MOD_PATH, TRAITS_NEW_VANILLA_PATH)
# run_and_print_diff("Siege Events (real files)", SIEGE_EVENTS_OLD_VANILLA_PATH, SIEGE_EVENTS_MOD_PATH, SIEGE_EVENTS_NEW_VANILLA_PATH)

# Run the benchmark test
run_benchmark_test("Comprehensive Benchmark", BENCHMARK_OLD, BENCHMARK_MOD, BENCHMARK_NEW, benchmark_assertions)

print("\n--- All Tests Complete ---")

--- Successfully imported PdsParser from: pds_parser.py ---
--- Successfully imported PdsDiffer from: pds_differ.py ---
--------------------------------------------------------------------------------
Running diff tests with real CK3 files.

==================== RUNNING TEST: 00_tribal_innovations (real files) ====================
Output files for this test will be saved to: c:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\test_output\00_tribal_innovations_real_files
  Raw files saved.
  Parsing Old file: 00_tribal_innovations.txt
  Parsing Mod file: 00_tribal_innovations.txt
  Parsing New file: 00_tribal_innovations.txt
  Reconstructed files saved.




--- DETECTED CHANGES for '00_tribal_innovations (real files)' (35 changes) ---
PdsChange(Type='VANILLA_MODIFIED                             ', Path='innovation_bannus___0', 
          ParentCtx='ROOT_PARENT', 
          Nodes=[
            O:PdsBlock L83 I0 innovation_bannus={...},
            M:PdsBlock L83 I0 innovation_bannus={...},
     